In [1]:
import os
from pathlib import Path

import pandas as pd
import time


def get_env_value(key, default=None, env_path=".env"):
    # Prefer shell variables first, then fall back to the local .env file so the notebook works in VS Code and headless runs.
    value = os.getenv(key)
    if value:
        return value

    if os.path.exists(env_path):
        with open(env_path, "r", encoding="utf-8") as env_file:
            for line in env_file:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                name, raw_value = line.split("=", 1)
                if name.strip() == key:
                    return raw_value.strip().strip('"').strip("'")
    return default


def get_env_int(key, default):
    return int(get_env_value(key, str(default)))


def get_env_float(key, default):
    return float(get_env_value(key, str(default)))


# Keep the notebook data paths configurable so the same code works across local machines and test setups.
csv_path = Path(get_env_value("CSV_PATH", "IOT Data Simulation/smart_logistic_tracker_japan.csv"))
sample_rows = get_env_int("SAMPLE_ROWS", 5)
write_delay_seconds = get_env_float("WRITE_DELAY_SECONDS", 0.1)
write_gas_limit = get_env_int("WRITE_GAS_LIMIT", 3000000)
enable_duplicate_writes = get_env_value("ENABLE_DUPLICATE_WRITES", "false").lower() in {"1", "true", "yes", "on"}
abi_path = Path(get_env_value("ABI_PATH", "contracts/abi.json"))

# Load the CSV file with basic error handling
try:
    df = pd.read_csv(csv_path)
    print(f"Total records in CSV: {len(df)}")
    print(f"First {sample_rows} records:")

    # Display the first few rows
    display(df.head(sample_rows))
except FileNotFoundError:
    print(f"❌ CSV file not found: {csv_path}")
    df = pd.DataFrame()
except pd.errors.EmptyDataError:
    print(f"❌ CSV file is empty: {csv_path}")
    df = pd.DataFrame()
except pd.errors.ParserError as error:
    print(f"❌ Failed to parse CSV file {csv_path}: {error}")
    df = pd.DataFrame()
except Exception as error:
    print(f"❌ Unexpected error while loading {csv_path}: {error}")
    df = pd.DataFrame()

Total records in CSV: 100
First 5 records:


,timestamp,carrier,tracking_number,package_id,origin,current_location,delivery_location,prefecture,latitude,longitude,...,waiting_time_minutes,perishable,temperature,humidity,rfid_tag,rfid_verified,tamper_alert,traffic_status,inventory_level,asset_utilization
0,2026-05-04 13:50:26.857905,Yamato Transport,942646961460,PKG7545,Tokyo,Naha Central Post Office,Tokyo,Kanagawa,35.993159,139.038781,...,45,No,10.7,40,RFID736892,False,No,Heavy,99,84.59
1,2026-05-03 23:36:26.858095,Japan Post,74355111775,PKG2659,Tokyo,Nagoya Central Post Office,Kyoto,Kanagawa,35.691292,139.130870,...,54,Yes,6.2,82,RFID156229,False,Yes,Detour,363,53.39
2,2026-05-04 08:32:26.858217,Japan Post,217497030475,PKG7965,Osaka,Nagoya Central Post Office,Osaka,Aichi,35.591109,139.784940,...,144,Yes,-3.1,86,RFID890703,True,Yes,Heavy,25,95.75
3,2026-05-04 02:35:26.858332,Japan Post,249781996688,PKG5296,Fukuoka,Sapporo Central Post Office,Sapporo,Osaka,35.570440,139.689163,...,173,Yes,6.3,87,RFID603182,True,Yes,Detour,145,63.84
4,2026-05-04 08:28:26.858444,Japan Post,718415724062,PKG9987,Fukuoka,Yokohama Sales Office,Sapporo,Hokkaido,35.679375,139.408071,...,82,No,0.7,60,RFID921432,False,Yes,Detour,34,56.28


In [2]:
from web3 import Web3

# Connect to local blockchain
ganache_url = get_env_value("GANACHE_URL", "http://127.0.0.1:8545")
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Verify connection
if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

✅ Connected to Ganache successfully!


In [3]:
import json

# Use the loaded ABI path and deployed contract address.
contract_address = get_env_value("CONTRACT_ADDRESS")
if not contract_address:
    raise ValueError("CONTRACT_ADDRESS is missing. Set it in .env or the environment.")
contract_address = Web3.to_checksum_address(contract_address)

# Load the ABI that matches the deployed contract in this repository.
with open(abi_path, "r", encoding="utf-8") as abi_file:
    abi = json.load(abi_file)

# Load the smart contract.
contract = web3.eth.contract(address=contract_address, abi=abi)

# Ganache may expose a different unlocked account set than the deployed contract owner,
# so we fall back to an explicit override when needed.
contract_owner = contract.functions.owner().call()
if contract_owner not in web3.eth.accounts:
    override_owner = get_env_value("CONTRACT_OWNER")
    if not override_owner:
        raise ValueError(
            f"Contract owner {contract_owner} is not unlocked in Ganache. "
            "Set CONTRACT_OWNER in .env to an unlocked account."
        )
    contract_owner = Web3.to_checksum_address(override_owner)
    if contract_owner not in web3.eth.accounts:
        raise ValueError(
            f"CONTRACT_OWNER {contract_owner} is not unlocked in Ganache."
        )

web3.eth.default_account = contract_owner

print(f"✅ Connected to Smart Contract at {contract_address}")
print(f"✅ Using sender account: {web3.eth.default_account}")

✅ Connected to Smart Contract at 0x7abf4b356FB67C8a9917c7E1E543895DB1Bf53b4
✅ Using sender account: 0x1C73Dd704ffeE88a4f4aAD5bA3B1af87C5884D0F


In [4]:
# Retrieve all existing records from the blockchain once at startup to cache them.
# This prevents expensive O(N) blockchain roundtrips for duplicate verification on every iteration.
total_records = contract.functions.getTotalRecords().call()
existing_records = set()
for record_index in range(total_records):
    record = contract.functions.getRecord(record_index).call()
    existing_records.add((str(record[1]), str(record[2]), str(record[3])))

def record_exists(package_id, data_type, data_value):
    """Return True when the exact record is already in the cache."""
    return (str(package_id), str(data_type), str(data_value)) in existing_records


def send_iot_data(package_id, data_type, data_value):
    """
    Sends logistics IoT data
    to the deployed smart contract
    """

    # Skip exact duplicates unless testing explicitly requires them.
    if not enable_duplicate_writes and record_exists(package_id, data_type, data_value):
        print(
            f"ℹ️ Skipped duplicate | {package_id} | "
            f"Type: {data_type} | Value: {data_value}"
        )
        return False

    if enable_duplicate_writes:
        print("ℹ️ Duplicate-write mode is ON; exact duplicates will be stored.")

    txn = contract.functions.storeData(
        package_id,
        data_type,
        data_value
    ).transact({
        'from': web3.eth.default_account,
        'gas': write_gas_limit
    })

    # Wait for transaction confirmation before moving to the next record.
    receipt = web3.eth.wait_for_transaction_receipt(txn)

    # Cache the new entry locally
    existing_records.add((str(package_id), str(data_type), str(data_value)))

    print(
        f"✅ Data Stored | {package_id} | "
        f"Type: {data_type} | "
        f"Value: {data_value} | "
        f"Txn Hash: {receipt.transactionHash.hex()}"
    )
    return True

# Each CSV row writes multiple contract entries depending on the columns.
is_iot_data = "shipment_id" in df.columns
entries_per_row = 3 if is_iot_data else 4

target_contract_records = int(get_env_value("TARGET_CONTRACT_RECORDS", "100"))
target_rows = target_contract_records // entries_per_row
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records
rows_to_store = min(len(df), target_rows, remaining_entries // entries_per_row)

print(f"Target contract records: {target_contract_records}")
print(f"Current records: {current_records}")
print(f"Maximum records: {max_entries}")
print(f"Remaining contract slots: {remaining_entries}")
print(f"Rows that can still be stored safely: {rows_to_store}")
print(f"Duplicate writes enabled: {enable_duplicate_writes}")

if target_contract_records % entries_per_row != 0:
    print(f"⚠️ TARGET_CONTRACT_RECORDS is not a multiple of {entries_per_row}, ignoring remaining slots to keep rows complete.")

if rows_to_store <= 0:
    print("⚠️ No remaining storage capacity on the contract.")
else:
    stored_rows = 0
    skipped_rows = 0

    for index, row in df.head(rows_to_store).iterrows():
        if is_iot_data:
            package_id = str(row["shipment_id"])
            status = str(row["shipment_status"])
            temp = f"{row['temperature']}°C"
            humid = f"{row['humidity']}%"

            s1 = send_iot_data(package_id, "Status", status)
            s2 = send_iot_data(package_id, "Temperature", temp)
            s3 = send_iot_data(package_id, "Humidity", humid)

            if s1 or s2 or s3:
                stored_rows += 1
            else:
                skipped_rows += 1
        else:
            package_id = str(row["package_id"])
            location = str(row["current_location"])
            status = str(row["latest_status"])
            temp = f"{row['temperature']}°C"
            humid = f"{row['humidity']}%"

            s1 = send_iot_data(package_id, "Location", location)
            s2 = send_iot_data(package_id, "Status", status)
            s3 = send_iot_data(package_id, "Temperature", temp)
            s4 = send_iot_data(package_id, "Humidity", humid)

            if s1 or s2 or s3 or s4:
                stored_rows += 1
            else:
                skipped_rows += 1

        # Small pause between transactions keeps Ganache logs readable and avoids flooding the provider.
        time.sleep(write_delay_seconds)

    print(f"\n✅ Successfully stored {stored_rows} new rows on the blockchain!")
    if skipped_rows:
        print(f"ℹ️ Skipped {skipped_rows} duplicate rows.")

Target contract records: 100
Current records: 265
Maximum records: 500
Remaining contract slots: 235
Rows that can still be stored safely: 25
Duplicate writes enabled: False
ℹ️ Skipped duplicate | PKG7545 | Type: Location | Value: Naha Central Post Office
ℹ️ Skipped duplicate | PKG7545 | Type: Status | Value: Out for Delivery
✅ Data Stored | PKG7545 | Type: Temperature | Value: 10.7°C | Txn Hash: 888a3c56f28bfaadf8d98844fc1bc700780561a5cd711eb8aedbe112376f84b7
✅ Data Stored | PKG7545 | Type: Humidity | Value: 40% | Txn Hash: 4e1253b03adf77419fbda46e9bcddb113e3d9b16a9571ef45558b9e45bbc08ec
ℹ️ Skipped duplicate | PKG2659 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG2659 | Type: Status | Value: Arrival
✅ Data Stored | PKG2659 | Type: Temperature | Value: 6.2°C | Txn Hash: 38c959b2f31990da742e531ba5db1019eb2408330b5a84a7741464c7bbbc3721


✅ Data Stored | PKG2659 | Type: Humidity | Value: 82% | Txn Hash: 6c67622075dfc68c6326e10cf5570f826ef89b6faba88480f6c259ba57229202
ℹ️ Skipped duplicate | PKG7965 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG7965 | Type: Status | Value: Storage
✅ Data Stored | PKG7965 | Type: Temperature | Value: -3.1°C | Txn Hash: 49242e733e9c2c1cd49481f0d6625dfadc48823e4231772b43f08578914a5b71
✅ Data Stored | PKG7965 | Type: Humidity | Value: 86% | Txn Hash: b8ab69e709eb4dc42864df6e52ee6bde6c320f28c4a0a839e8abc428841beede


ℹ️ Skipped duplicate | PKG5296 | Type: Location | Value: Sapporo Central Post Office
ℹ️ Skipped duplicate | PKG5296 | Type: Status | Value: Delivered to the delivery address
✅ Data Stored | PKG5296 | Type: Temperature | Value: 6.3°C | Txn Hash: 57a4e9043303b5f4f405aac9a752e332f8ffc8158110390b1a1dad2212bf392a
✅ Data Stored | PKG5296 | Type: Humidity | Value: 87% | Txn Hash: 67eec4f9146f39cb6a5ef140a6c20997b62a18d7ebc99fe44838f7890e07257d
ℹ️ Skipped duplicate | PKG9987 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG9987 | Type: Status | Value: Bring it back due to your absence
✅ Data Stored | PKG9987 | Type: Temperature | Value: 0.7°C | Txn Hash: c47a0de9c210e0d43176d84ad7a25cc40e7909fd3a7de9712f7b3e160499ee52
✅ Data Stored | PKG9987 | Type: Humidity | Value: 60% | Txn Hash: 7732f0697619dfb7b04e5ee262a59c58b7a02ae0d408d42d42627b893a65b2a0


ℹ️ Skipped duplicate | PKG8392 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG8392 | Type: Status | Value: Hold at Yamato
✅ Data Stored | PKG8392 | Type: Temperature | Value: 2.5°C | Txn Hash: bd27779f44ffb2c7b77163ff1da91f3d04c37a38d4f39a0b802466f9ddde4b4e
✅ Data Stored | PKG8392 | Type: Humidity | Value: 61% | Txn Hash: 5de517f64776dc5627876e39cb64b7bc8f8b36324916c59227c89f46f9f8bea4
ℹ️ Skipped duplicate | PKG2808 | Type: Location | Value: Naha Central Post Office
ℹ️ Skipped duplicate | PKG2808 | Type: Status | Value: Delay
✅ Data Stored | PKG2808 | Type: Temperature | Value: 0.9°C | Txn Hash: f8b00470e187b1d8349cd20640aa85a7f1f51e7bc485a2178d5368e800fcc5d2
✅ Data Stored | PKG2808 | Type: Humidity | Value: 48% | Txn Hash: db2c843f13a1b8762d5f90ae6c9c12724c7d21c9f709f4d4980accb8737f17b6


ℹ️ Skipped duplicate | PKG1749 | Type: Location | Value: Fukuoka Distribution Center
ℹ️ Skipped duplicate | PKG1749 | Type: Status | Value: Arrival
✅ Data Stored | PKG1749 | Type: Temperature | Value: 19.6°C | Txn Hash: 21bd33c3227aa94fbbede9083b1225277bc3eb40c3d865d7a84640ae3fdad165
✅ Data Stored | PKG1749 | Type: Humidity | Value: 34% | Txn Hash: cffe4af08425b654f9b28609c762c52fe1278b67c33a3b2e874b4da1ad744758
ℹ️ Skipped duplicate | PKG1803 | Type: Location | Value: Sapporo Central Post Office
ℹ️ Skipped duplicate | PKG1803 | Type: Status | Value: Hand it over at the window
✅ Data Stored | PKG1803 | Type: Temperature | Value: 22.6°C | Txn Hash: 08a23c5f21b1fa6083770e73a45c2f69fa8215298f2766185fa979fa2cea92c4
✅ Data Stored | PKG1803 | Type: Humidity | Value: 55% | Txn Hash: 5f80f3ceee42376d0e1474e992e32b0d53f2ea34ec3e916e20408177f4c92b6c


...

ℹ️ Skipped duplicate | PKG1347 | Type: Location | Value: Nagoya Central Post Office
ℹ️ Skipped duplicate | PKG1347 | Type: Status | Value: Returned to the sender
✅ Data Stored | PKG1347 | Type: Temperature | Value: 18.8°C | Txn Hash: e49af516fde58570487df0caccb50d522535d2d3ad76faef93acb681e38de1aa
✅ Data Stored | PKG1347 | Type: Humidity | Value: 76% | Txn Hash: 65f562ccd68375d0ee6d197f3d81536f03841fc0423cbc5d2f6b19f8025cff64
ℹ️ Skipped duplicate | PKG8088 | Type: Location | Value: Tokyo Central Post Office
ℹ️ Skipped duplicate | PKG8088 | Type: Status | Value: Under Investigation
✅ Data Stored | PKG8088 | Type: Temperature | Value: 8.7°C | Txn Hash: ee8779063a49a187afe6659babd4c06f51038878ae6376e17fe2e8945674d16a
✅ Data Stored | PKG8088 | Type: Humidity | Value: 37% | Txn Hash: bf2094e6118bc35caee7c0da7227f9dec27c64a7447661a98c148a970562a9e6


ℹ️ Skipped duplicate | PKG2762 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG2762 | Type: Status | Value: Arrival Scan
✅ Data Stored | PKG2762 | Type: Temperature | Value: -2.5°C | Txn Hash: 9cb510a5a1327f085581fa8033fba15cef04d3fdf82e724e7c3fb59c56b8a9a5
✅ Data Stored | PKG2762 | Type: Humidity | Value: 77% | Txn Hash: 224c705468aab866d3dab35f0ac556423aca9fa065b75984b41e6cb8f411c3f8
ℹ️ Skipped duplicate | PKG8577 | Type: Location | Value: Naha Central Post Office
ℹ️ Skipped duplicate | PKG8577 | Type: Status | Value: Hand it over at the window
✅ Data Stored | PKG8577 | Type: Temperature | Value: 0.7°C | Txn Hash: 93525d351b669c3f39d48c6e19a84c8e79ff87441a5a0c4bd48570b864072ad5
✅ Data Stored | PKG8577 | Type: Humidity | Value: 36% | Txn Hash: 34cf148f1bdb2ca6c4ace1591a32edddfb7ef37e0b56f0d5cb9ebd1c7c8c9f48


ℹ️ Skipped duplicate | PKG5517 | Type: Location | Value: Yokohama Sales Office
ℹ️ Skipped duplicate | PKG5517 | Type: Status | Value: Delay
✅ Data Stored | PKG5517 | Type: Temperature | Value: -2.4°C | Txn Hash: 5091b2b016ccb3ab35aebfc1b702469008c30059c169f5a83db3cf3048badfa8
✅ Data Stored | PKG5517 | Type: Humidity | Value: 85% | Txn Hash: 1a56dead59873c24447399b2c9711667faec6760864d8cfea3dab0c2664d2203
ℹ️ Skipped duplicate | PKG7409 | Type: Location | Value: Sapporo Central Post Office
ℹ️ Skipped duplicate | PKG7409 | Type: Status | Value: Arrival
✅ Data Stored | PKG7409 | Type: Temperature | Value: 2.3°C | Txn Hash: b3429a41966c7e84097f9a3f09f28ec3eb9d5ff33cdad102b223aeb012bb672a


✅ Data Stored | PKG7409 | Type: Humidity | Value: 65% | Txn Hash: 52f81c930c0b60a4cebf22f917ea6834ef52083fe56043cfbda8244dc0950a1b
ℹ️ Skipped duplicate | PKG5985 | Type: Location | Value: Tokyo Central Post Office
ℹ️ Skipped duplicate | PKG5985 | Type: Status | Value: Returned
✅ Data Stored | PKG5985 | Type: Temperature | Value: 4.3°C | Txn Hash: 14bfae96f753e9678ba8672edd78c515feecc1cf0ca879257ba8b8c982a572f2
✅ Data Stored | PKG5985 | Type: Humidity | Value: 61% | Txn Hash: e4a28b1359df901e5c562f1693cb7e100d7a9f80bd09dd5efec96a827a6ce556



✅ Successfully stored 25 new rows on the blockchain!


In [5]:
current_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {current_records}")

Total IoT records stored: 315


In [6]:
current_records = contract.functions.getTotalRecords().call()
max_entries = contract.functions.MAX_ENTRIES().call()
remaining_entries = max_entries - current_records

print(f"Current IoT records stored: {current_records}")
print(f"Maximum records allowed: {max_entries}")
print(f"Remaining storage slots: {remaining_entries}")

if remaining_entries == 0:
    print("⚠️ The contract is full. Redeploy a new contract or reset the chain to store more data.")
elif remaining_entries <= 20:
    print("⚠️ The contract is nearing capacity.")

Current IoT records stored: 315
Maximum records allowed: 500
Remaining storage slots: 185


In [7]:
# Retrieve and display the first stored record
first_record = contract.functions.getRecord(0).call()

print("📦 First Stored Record")
print(f"Timestamp: {first_record[0]}")
print(f"Package ID: {first_record[1]}")
print(f"Data Type: {first_record[2]}")
print(f"Data Value: {first_record[3]}")

📦 First Stored Record
Timestamp: 1780192485
Package ID: PKG7545
Data Type: Location
Data Value: Naha Central Post Office
